# Reproducing Schwab et al. (2020) from dandiset 000947

**Paper.** Schwab BC, Kase D, Zimnik A, Rosenbaum R, Codianni MG, Rubin JE, Turner RS (2020). Neural
activity during a simple reaching task in macaques is counter to gating and rebound in basal
ganglia-thalamic communication. *PLOS Biology* 18(10): e3000829.
[doi:10.1371/journal.pbio.3000829](https://doi.org/10.1371/journal.pbio.3000829). The paper is open
access and this attempt used its full text.

**Dandiset.** [DANDI:000947](https://dandiarchive.org/dandiset/000947/0.240510.2211), version
0.240510.2211. The paper predates the dandiset and cites no DANDI version (its data statement points
to Dryad). 0.240510.2211 is the only published version, so it is pinned; the draft lists the same
378 files.

**What the dandiset holds.** Single-unit recordings from two monkeys (Gaia, "NHP G", and Isis,
"NHP I") in GPi, VL thalamus, motor cortex and other areas, before and after MPTP, with trial
tables and raw and filtered traces. The paper uses the pre-MPTP GPi and VLa recordings. EMG,
the stimulation mapping of Fig 1 and the simulations of Fig 6 are not in the dandiset.

**Outside inputs.** The paper's Dryad deposit (doi:10.5061/dryad.0cfxpnvxm, read here from its
Zenodo mirror, record 4000719) holds the analysis code read for the method and the per-unit result
files `Gaia_Analy.mat` and `Isis_Analy.mat`. Those files name each analysed unit by file, channel
and sort letter; they are used only by the claim `unit-link` and by variants that use the paper's
own unit list. Every outcome on the paper's findings is decided on a unit selection made from the
dandiset alone.

| Claim | Data | Status |
|---|---|---|
| unit-link | with inputs | tested |
| table1-unit-counts | dandiset | tested |
| table1-rest-rate | dandiset | tested |
| table1-ap-width | dandiset | tested |
| hold-period-ramps | dandiset | tested |
| mvt-responsive-fraction | dandiset | tested |
| increases-most-common | dandiset | tested |
| monophasic-more-in-vla | dandiset | tested |
| integrated-change-positive | dandiset | tested |
| fig3a-vla-leads-gpi | dandiset | tested |
| fig3b-gpi-decr-lag-vla-incr | dandiset | tested |
| fig3c-vla-decr-lead-gpi-incr | dandiset | tested |
| gating-conflict-fraction | dandiset | tested |
| directional-units | dandiset | tested |
| fig4-ccf-rare | dandiset | tested |
| fig4-noise-corr-rare | dandiset | tested |
| fig5-vla-decrease-after-gpi-bursts | dandiset | tested |
| behavior-rt-mt | dandiset | tested |
| figs3-emg-onset | elsewhere | blocked: methods underspecified |
| fig1-vla-localization | dandiset | blocked: data missing |
| fig6-simulation | elsewhere | blocked: other |

In [ ]:
import io
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import scipy.io as sio
import statsmodels.api as sm
from dandi.dandiapi import DandiAPIClient
from scipy import stats
from scipy.ndimage import gaussian_filter1d
from statsmodels.formula.api import ols

import gpivla as g

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 300)
warnings.filterwarnings("ignore", category=RuntimeWarning)

NB_START = started = time.time()
all_files = g.asset_urls(pre_only=False)
sessions = g.load_all()
print(f"{len(all_files)} files in dandiset {g.DANDISET_ID} version {g.DANDISET_VERSION}; "
      f"loaded the {len(sessions)} pre-MPTP files in {time.time() - started:.0f} s")

## Data check

Each check of the metadata against the values is stated under its output.

In [ ]:
with DandiAPIClient() as client:
    meta = client.get_dandiset(g.DANDISET_ID, g.DANDISET_VERSION).get_raw_metadata()
summary = meta["assetsSummary"]
print("Archive summary: subjects", summary["numberOfSubjects"], "| files", summary["numberOfFiles"],
      "| species", [s["name"] for s in summary["species"]])
print("Related resources:", [r.get("identifier") or r.get("url") for r in meta.get("relatedResource", [])])
paths = pd.DataFrame({"path": [p for p, _ in all_files]})
paths["subject"] = paths.path.str.extract(r"sub-(\w+)/")[0]
paths["condition"] = np.where(paths.path.str.contains("ses-pre-MPTP"), "pre-MPTP", "post-MPTP")
print("\nFiles by subject and condition:")
print(pd.crosstab(paths.subject, paths.condition).to_string())

info = pd.DataFrame([{
    "session": s["session_id"], "key": s["key"], "subject": s["subject"], "mptp_attr": s["mptp"],
    "start": s["session_start"][:10], "has_trials": s["trials"] is not None,
    "n_trials": 0 if s["trials"] is None else len(s["trials"]), "n_units": len(s["units"]),
    "publications": ";".join(s["related_publications"]), "groups": str(sorted(s["groups"].values()))}
    for s in sessions])
print("\nMPTP status attribute of the pre-MPTP files:", info.mptp_attr.value_counts().to_dict())
print("related_publications in the files:", info.publications.value_counts().to_dict())
print("Files without a trials table:", info.session[~info.has_trials].tolist())
print("Files without curated units:", int((info.n_units == 0).sum()), "of", len(info))
print("Trials per file: min", info.n_trials[info.has_trials].min(), "median",
      info.n_trials[info.has_trials].median(), "max", info.n_trials.max())
day = pd.to_datetime(info.session.str.extract(r"-(\d{6})-")[0], format="%y%m%d")
offset = (pd.to_datetime(info.start) - day).dt.days
print("session_start_time minus the date in session_id (days):", offset.value_counts().to_dict())
print("Electrode group locations (files):")
print(info.groups.value_counts().head(12).to_string())

The archive summary counts 2 subjects, as the files do (Gaia and Isis). Every pre-MPTP file
carries the MPTP status `pre_MPTP`, consistent with its name. The files name the paper only by its
bioRxiv preprint (doi:10.1101/546598); the post-MPTP files name a different preprint. One
pre-MPTP file has no trials table and no curated units. In every file the
`session_start_time` is two days after the date in the `session_id`; the paper does not use dates,
so this does not affect the analysis, but one of the two is wrong or shifted on purpose and the
files do not say which.

In [ ]:
rows = []
for s in sessions:
    span = None
    for u in s["units"]:
        sp = u["spikes"]
        rows.append({"session": s["session_id"], "key": s["key"], "subject": s["subject"],
                     "unit_name": u.get("unit_name"), "label": u.get("location"),
                     "area": g.area(u.get("location")), "channel": pd.to_numeric(u.get("channel_ids"), errors="coerce"),
                     "sort": str(u.get("sort_label")), "quality": u.get("unit_quality_post_sorting"),
                     "n_spikes": len(sp), "first": sp[0] if len(sp) else np.nan, "last": sp[-1] if len(sp) else np.nan,
                     "sorted": u["sorted_in_file"], "rec_s": s["n_raw"] / 24414.0625})
U = pd.DataFrame(rows)
print("Curated units by subject and location label as written in the files:")
print(pd.crosstab(U.label.fillna("(none)").replace("", "(empty)"), U.subject).to_string())
print("\nUnits with no channel id:", int(U.channel.isna().sum()), "; with an empty location:", int((U.label.fillna("") == "").sum()))
print("Sort labels:", U["sort"].value_counts().to_dict())
print("Quality labels (GPi and VLa):", U[U.area.isin(["GPi", "VLa"])].quality.fillna("(column absent)").value_counts().to_dict())
print("\nSpike times stored in increasing order in every unit:", bool(U.sorted.all()))
print("Spike times within the raw recording (0 to its length):",
      bool(((U["first"] >= 0) & (U["last"] <= U.rec_s + 1)).all()))
# Recordings deposited twice: identical spike trains within or across files.
digest = {}
for s in sessions:
    for u in s["units"]:
        if len(u["spikes"]):
            digest.setdefault((len(u["spikes"]), round(float(u["spikes"][0]), 6), round(float(u["spikes"][-1]), 6)), []).append(
                (s["session_id"], u.get("unit_name")))
dups = {k: v for k, v in digest.items() if len(v) > 1}
print("Spike trains with the same count, first and last spike in more than one unit:", len(dups))
for k, v in list(dups.items())[:10]:
    us = [u for s in sessions if s["session_id"] == v[0][0] for u in s["units"] if u.get("unit_name") in [x[1] for x in v]]
    print("  ", v, "labels", [u.get("location") for u in us], "channels", [u.get("channel_ids") for u in us],
          "identical:", all(np.array_equal(us[0]["spikes"], x["spikes"]) for x in us[1:]))
same_name = U.groupby(["session", "unit_name"]).size()
print("Unit names used twice within a file:", int((same_name > 1).sum()))

The location labels differ in case between files (GPi and GPI, VLa and VLA), so they are compared
here ignoring case. In Gaia, 54 thalamic units are labelled only "Thalamus", and in Isis 33 units
have an empty label and no channel id; the paper's code calls many of these VLa (see `unit-link`),
but the dandiset does not say so, and the dandiset rule leaves them out. Sort labels are written as
letters in some files and as numbers in others, and the quality column is missing from some files
and holds free text in others. Unit names repeat within some files (a unit on the GPi probe and a
unit with an empty label share a name). One spike train is stored twice, as two M1-S units on
different channels of one file; no GPi or VLa train is, so no recording used here is counted twice
by identity. A neuron recorded over consecutive files appears once per file (see
`table1-unit-counts`).

In [ ]:
for s in sessions:
    s["vt"] = None if s["trials"] is None else g.valid_trials(s["trials"])
tr = pd.concat([s["vt"].assign(session=s["session_id"], subject=s["subject"]) for s in sessions if s["vt"] is not None])
print("Trial table columns (files):")
print(pd.Series([tuple(s["trials"].columns) for s in sessions if s["trials"] is not None]).value_counts().to_string())
print("\nTrials by subject: total, errors, outliers, valid")
print(tr.groupby("subject").agg(total=("valid", "size"), errors=("error", "sum"),
                                outliers=("outlier", "sum"), valid=("valid", "sum")).to_string())
print("Targets:", tr.target.value_counts().to_dict())
v = tr[tr.valid]
print("Reaction time of valid trials (s): min %.3f median %.3f max %.3f" % (v.rt.min(), v.rt.median(), v.rt.max()))
print("Movement duration of valid trials (s): min %.3f median %.3f max %.3f; trials without one: %d of %d"
      % (v.mt.min(), v.mt.median(), v.mt.max(), v.mt.isna().sum(), len(v)))
order_ok = (tr.cue_onset_time <= tr.movement_start_time) | ~tr.valid
print("Valid trials with cue onset before movement onset:", bool(order_ok.all()))
print("Files with movement ends, by subject:",
      {k: int(x) for k, x in pd.Series({s["session_id"]: ("movement_stop_time" in s["trials"]) for s in sessions if s["trials"] is not None}).groupby(lambda k: k.split("-")[2]).sum().items()})

Times are in seconds, and reaction times and movement durations of valid trials fall in the range
the task allows (the animal had 1 s to reach). The paper's reaction times (S2 Fig) are around
0.2 to 0.3 s, as here. Error trials are flagged only in files that have an `error_onset_time`
column; in the other files no trial is marked as an error, so error trials there, if any, cannot
be excluded. Movement ends are missing from many files, so movement durations are available for
part of the data only.

**Clock check.** With spikes and a trial table only, the check is whether the trial labels carry
information in the spikes at the time of movement but not before the cue, when the animal does not
know the target. For each unit with at least 10 valid trials per direction, left and right spike
counts are compared (rank sum) in the movement window (-200 to +300 ms) and in a window before the
cue (-1500 to -1000 ms from movement onset). Two controls: the labels permuted within the file,
and the true labels with every trial time moved 1 s later (a clock offset).

In [ ]:
def window_counts(spikes, align, lo, hi):
    a = np.searchsorted(spikes, align + lo)
    b = np.searchsorted(spikes, align + hi)
    return b - a

rng = np.random.default_rng(0)
rows = []
for s in sessions:
    if s["vt"] is None:
        continue
    t = s["vt"][s["vt"].valid]
    left = (t.target == "Left").to_numpy()
    permuted = rng.permutation(left)
    al = t.movement_start_time.to_numpy()
    for u in s["units"]:
        if g.area(u.get("location")) not in ("GPi", "VLa") or left.sum() < 10 or (~left).sum() < 10:
            continue
        r = {"session": s["session_id"], "subject": s["subject"], "area": g.area(u.get("location"))}
        for name, (lo, hi) in {"movement": (-0.2, 0.3), "pre_cue": (-1.5, -1.0)}.items():
            for lab, mask, shift in (("true", left, 0.0), ("permuted", permuted, 0.0), ("offset_1s", left, 1.0)):
                c = window_counts(u["spikes"], al + shift, lo, hi)
                r[f"{name}_{lab}"] = stats.mannwhitneyu(c[mask], c[~mask]).pvalue if c.std() > 0 else 1.0
        rows.append(r)
clock = pd.DataFrame(rows)
cols = [f"{w}_{l}" for w in ("movement", "pre_cue") for l in ("true", "permuted", "offset_1s")]
print("Fraction of units whose spike counts differ between left and right trials (p < 0.05):")
print(clock.groupby("area")[cols].agg(lambda p: (p < 0.05).mean()).round(3).T.to_string())
per_session = clock.groupby("session").movement_true.apply(lambda p: (p < 0.05).any())
print(f"\nFiles with at least one direction-tuned unit in the movement window: {per_session.sum()} of {len(per_session)}")
print("Files where none is tuned (the clock cannot be checked this way):", sorted(per_session[~per_session].index))

With the true labels, 48.5% of GPi and 38.6% of VLa units differ between left and right trials in
the movement window, against 8.7% and 7.8% before the cue; with permuted labels both windows are
near the 5% expected by chance (3.4% to 6.7%). The 1 s offset does not remove the tuning (37.2% and
31.9%), because activity while holding at the target also depends on the target, so it does not
test the clock. The contrast between the movement and pre-cue windows is consistent with the spike
times and the trial table sharing a clock; the detected response onsets below, with medians
between -84 and -19 ms from movement onset, support it. In 34 of the 163 files with GPi or
VLa units no unit is tuned, and for them the shared clock is assumed, not shown.

## Unit selection and peri-movement responses

The dandiset rule (fixed in the claims file): curated units labelled GPi or VLa (ignoring case) in
pre-MPTP files; a unit-direction is analysed with at least 10 valid trials; a unit is kept when its
mean rate is at least 30 spikes/s (GPi) or 1 spike/s (VLa) and at least one direction is analysed.
Each file's unit is one unit. For every unit-direction the helper computes the SDF and ISI
functions, detects increases and decreases, and records the rest rate, the integrated change and
the control-period slope.

In [ ]:
started = time.time()
def run_units(records):
    """records: list of dicts with unit id, area, subject and segments [(spikes, valid trials)]."""
    out = []
    for r in records:
        for target in ("Left", "Right"):
            d = g.unit_direction(r["segments"], target)
            if d is None:
                continue
            out.append({"uid": r["uid"], "area": r["area"], "subject": r["subject"], "target": target,
                        **{k: d[k] for k in ("n_trials", "median_rt_ms", "rest_rate", "integrated", "inc_onset",
                                             "dec_onset", "slope_per_s2", "slope_sig")},
                        "sdf": d["sdf"], "counts": d["trial_counts"]})
    df = pd.DataFrame(out)
    df["form"] = [g.response_form(i, d) for i, d in zip(df.inc_onset, df.dec_onset)]
    return df

dand_records, dropped = [], {"rate": 0, "trials": 0}
for s in sessions:
    if s["vt"] is None:
        continue
    for k, u in enumerate(s["units"]):
        a = g.area(u.get("location"))
        if a not in ("GPi", "VLa") or len(u["spikes"]) < 2:
            continue
        rate = len(u["spikes"]) / (u["spikes"][-1] - u["spikes"][0])
        if rate < g.MIN_RATE[a]:
            dropped["rate"] += 1
            continue
        dand_records.append({"uid": f"{s['session_id']}/{k}", "area": a, "subject": s["subject"], "session": s["session_id"],
                             "key": s["key"], "channel": pd.to_numeric(u.get("channel_ids"), errors="coerce"),
                             "sort": str(u.get("sort_label")), "rate": rate, "segments": [(u["spikes"], s["vt"])],
                             "unit": u, "sess": s})
ud = run_units(dand_records)
kept = set(ud.uid)
dropped["trials"] = sum(r["uid"] not in kept for r in dand_records)
units = pd.DataFrame([{k: r[k] for k in ("uid", "area", "subject", "session", "key", "channel", "sort", "rate")}
                      for r in dand_records if r["uid"] in kept])
print(f"Dandiset rule: {len(units)} units, {len(ud)} unit-directions ({time.time() - started:.0f} s)")
print("Left out: below the rate criterion", dropped["rate"], "; fewer than 10 valid trials in both directions", dropped["trials"])
print(units.groupby(["area", "subject"]).size().unstack().to_string())

The paper's unit list, for the variants: the records the paper's code keeps (`Fig2_3_S8_S9.m`
drops VLp units and records with missing SDFs) are matched to the curated units by file, channel id
and sort letter; a record that spans several files pools the trials of all of them, as the paper's
records do. Records not found in every file they span are left out of the variants.

In [ ]:
started = time.time()
ZENODO = "https://zenodo.org/records/4000719/files/{}?download=1"
analy = {}
for monk in ("Gaia", "Isis"):
    r = requests.get(ZENODO.format(f"{monk}_Analy.mat"), timeout=600)
    r.raise_for_status()
    m = sio.loadmat(io.BytesIO(r.content), squeeze_me=True, struct_as_record=False)
    analy[monk] = (m["RespSDFIncr"], m["RespISIIncr"])

def sort_letter(x):
    return {"1.0": "a", "2.0": "b", "3.0": "c", "4.0": "d"}.get(str(x), str(x))

by_key = {s["key"]: s for s in sessions}
paper_rows, paper_records = [], []
for monk, (A, B) in analy.items():
    for i in range(len(A.fname)):
        utype = str(A.unit_type[i])
        files = str(A.fname[i]).split("/")
        keep = utype != "VLp" and not np.isnan(A.evt_sdf[i, :, 0]).any()
        found = []
        for f in files:
            s = by_key.get(f)
            hits = [] if s is None else [u for u in s["units"]
                    if pd.to_numeric(u.get("channel_ids"), errors="coerce") == float(A.Uchan[i])
                    and sort_letter(u.get("sort_label")) == A.Usort[i]]
            found.append((s, hits[0]) if len(hits) == 1 else None)
        complete = all(x is not None for x in found)
        paper_rows.append({"monk": monk, "i": i, "fname": str(A.fname[i]), "area": utype, "keep": keep,
                           "n_files": len(files), "n_found": sum(x is not None for x in found), "complete": complete,
                           "labels": sorted({g.area(x[1].get("location")) or "(empty)" for x in found if x})})
        if keep and complete:
            paper_records.append({"uid": f"paper/{monk}/{i}", "area": utype, "subject": monk, "monk": monk, "i": i,
                                  "segments": [(u["spikes"], s["vt"]) for s, u in found]})
paper = pd.DataFrame(paper_rows)
ud_paper = run_units(paper_records)
print(f"Paper's records: {len(paper)}; kept by its code: {int(paper.keep.sum())}; found in every file: "
      f"{int((paper.keep & paper.complete).sum())}; unit-directions analysed: {len(ud_paper)} ({time.time() - started:.0f} s)")

## unit-link

> The units the paper analysed can be found in the dandiset, and the dandiset's spike times and
> trials reproduce the per-unit peri-movement activity the paper's code stored.

In [ ]:
k = paper[paper.keep]
print("Records kept by the paper's code, by animal and area:")
print(k.groupby(["monk", "area"]).size().unstack().to_string())
print("\nFound in every file they span:")
print(k.groupby(["monk", "area"]).complete.agg(["sum", "size", "mean"]).round(3).to_string())
print(f"Overall: {int(k.complete.sum())} of {len(k)} ({k.complete.mean():.1%}); records spanning 2 to 5 files: {int((k.n_files > 1).sum())}")
missing = k[~k.complete]
print("\nRecords not found, by file:")
print(missing.groupby("fname").size().to_string())
print("\nDandiset labels of the matched records, by the paper's area:")
print(pd.crosstab(k[k.complete].area, k[k.complete].labels.map(lambda l: "/".join(l))).to_string())

rows = []
for _, r in ud_paper.iterrows():
    monk, i = r.uid.split("/")[1], int(r.uid.split("/")[2])
    A, B = analy[monk]
    t = 0 if r.target == "Left" else 1
    ref = A.evt_sdf[i, t, :]
    rows.append({"area": r.area, "r": np.corrcoef(r.sdf, ref)[0, 1], "mean_ratio": r.sdf.mean() / ref.mean(),
                 "n": r.n_trials, "n_ref": np.shape(A.evt_spkt[i, t])[0],
                 "inc": r.inc_onset, "inc_ref": A.on_ind[i, t] - 1500, "dec": r.dec_onset, "dec_ref": B.on_ind[i, t] - 1500})
cmp_ = pd.DataFrame(rows)
print(f"\nUnit-directions compared: {len(cmp_)}")
print(f"SDF correlation with the stored SDF: median {cmp_.r.median():.4f}; r > 0.9 for {(cmp_.r > 0.9).mean():.1%}; "
      f"r > 0.99 for {(cmp_.r > 0.99).mean():.1%}; mean rate ratio median {cmp_.mean_ratio.median():.4f}")
print(f"Valid trial counts equal to the paper's: {(cmp_.n == cmp_.n_ref).mean():.1%}; "
      f"median absolute difference {np.median(np.abs(cmp_.n - cmp_.n_ref)):.0f}, max {np.max(np.abs(cmp_.n - cmp_.n_ref)):.0f}")
for kind in ("inc", "dec"):
    a, b = cmp_[kind].notna(), cmp_[kind + "_ref"].notna()
    both = a & b
    print(f"{kind}reases: detected here {a.sum()}, by the paper {b.sum()}, agreement on presence {(a == b).mean():.1%}; "
          f"onset difference where both, median absolute {np.median(np.abs(cmp_[kind][both] - cmp_[kind + '_ref'][both])):.0f} ms")

**Result: reproduced.** Of the 394 records the paper's code keeps, 364 (92.4%) are found among the
dandiset's curated units in every file they span (Isis GPi 104 of 104, Gaia GPi 95 of 105, Gaia VLa
60 of 66, Isis VLa 105 of 119). For the 724 matched unit-directions, the SDF computed from the
dandiset correlates with the stored one at a median r of 0.9992, above 0.9 for 94.6%, with a median
rate ratio of 1.0003. Valid trial counts equal the paper's for 62.6% of unit-directions (median
difference 0). Detection agrees on the presence of an increase for 90.9% and of a decrease for
80.7% of unit-directions, with median onset differences of 7 and 11 ms. Of the matched units the
paper calls VLa, 25 are labelled "Thalamus" in the dandiset; 97 records span 2 to 5 files, which
the dandiset does not link.

## table1-unit-counts

> 209 single units met the criteria to be included as GPi neurons (105 in NHP G, 104 in NHP I)
> and 185 as VLa neurons (66 and 119).

In [ ]:
counts = units.groupby(["area", "subject"]).size().unstack()
counts["total"] = counts.sum(axis=1)
print("Dandiset rule (one unit per file):")
print(counts.to_string())
for a, ref in (("GPi", [209]), ("VLa", [185, 218])):
    n = int(counts.loc[a, "total"])
    print(f"{a}: {n}; paper {ref}; within 10%: {[abs(n - x) <= 0.1 * x for x in ref]}")
day = units.session.str.extract(r"-(\d{6})-")[0]
distinct = units.assign(day=day).dropna(subset=["channel"]).drop_duplicates(["subject", "day", "area", "channel", "sort"])
print("\nVariant, units with the same channel and sort letter on the same day taken as one neuron (an assumption):")
print(distinct.groupby(["area", "subject"]).size().unstack().assign(total=lambda d: d.sum(axis=1)).to_string())
print("\nVariant, the paper's records found in the dandiset:")
print(paper[paper.keep & paper.complete].groupby(["area", "monk"]).size().unstack().assign(total=lambda d: d.sum(axis=1)).to_string())
in_paper_file = units.key.isin({f for x in paper.fname for f in x.split("/")})
print(f"\nDandiset-rule units in files the paper used: {int(in_paper_file.sum())} of {len(units)}")

**Result: not reproduced.** The dandiset rule gives 268 GPi units (Gaia 137, Isis 131) and 279 VLa
units (78 and 201), against 209 and 185 (or 218 in the text); neither is within 10%. Counting
units with the same channel and sort letter on the same day once gives 132 GPi and 186 VLa units,
and the paper's own records found in the dandiset are 199 and 165. The dandiset holds more
GPi and VLa-labelled units than the paper analysed (518 of the 547 are in files the paper used) and
does not say which ones the paper kept, nor which units in consecutive files are one neuron.

## table1-rest-rate

> The resting firing rate of GPi neurons (mean 70.1 spikes/s) is significantly higher than that of
> VLa neurons (13.5 spikes/s), and the rates are consistent between the two animals.

In [ ]:
def rest_table(df):
    t = df.groupby(["area", "subject"]).rest_rate.agg(["mean", "std", "size"])
    t2 = df.groupby("area").rest_rate.agg(["mean", "std", "size"]).assign(subject="all").set_index("subject", append=True)
    return pd.concat([t, t2]).sort_index()

print("Rest rate (spikes/s), mean SDF from -1500 to -301 ms, over unit-directions. Dandiset rule:")
print(rest_table(ud).round(2).to_string())
gp, vl = ud[ud.area == "GPi"].rest_rate, ud[ud.area == "VLa"].rest_rate
res = stats.ranksums(gp, vl)
print(f"GPi {gp.mean():.2f} against VLa {vl.mean():.2f}; rank sum z = {res.statistic:.2f}, p = {res.pvalue:.2g}")
for a in ("GPi", "VLa"):
    x = ud[ud.area == a]
    p = stats.ranksums(x[x.subject == "Gaia"].rest_rate, x[x.subject == "Isis"].rest_rate).pvalue
    print(f"{a}: Gaia against Isis, rank sum p = {p:.3g}")
print("\nRange of rest rates (spikes/s):", ud.groupby("area").rest_rate.agg(["min", "max"]).round(1).to_dict("index"))
print("\nVariant, the paper's records:")
print(rest_table(ud_paper).round(2).to_string())
print(f"rank sum p = {stats.ranksums(ud_paper[ud_paper.area == 'GPi'].rest_rate, ud_paper[ud_paper.area == 'VLa'].rest_rate).pvalue:.2g}")

**Result: reproduced.** The mean rest rate over unit-directions is 75.65 spikes/s in GPi (SD 25.63,
531) and 12.78 in VLa (SD 11.62, 552), both within 10% of 70.1 and 13.5, and GPi is higher (rank
sum p = 1.2e-171). The animals differ slightly in GPi (73.17 and 78.25 spikes/s, p = 0.041), where
the paper reports no difference, and not in VLa (p = 0.53). With the paper's records the means are
64.26 and 13.53 spikes/s.

## table1-ap-width

> Action potentials of GPi neurons are shorter than those of VLa neurons (trough to peak 0.24
> against 0.50 ms; rank sum p = 1e-33).

In [ ]:
from concurrent.futures import ThreadPoolExecutor
started = time.time()
jobs = {}
for r in dand_records:
    if r["uid"] not in kept or np.isnan(r["channel"]):
        continue
    s = r["sess"]
    rows_ = np.flatnonzero(pd.to_numeric(s["electrodes"].channel_ids, errors="coerce").to_numpy() == r["channel"])
    for name, p in s["processed"].items():
        cols = np.flatnonzero(np.isin(p["electrodes"], rows_))
        if len(cols) == 1:
            jobs.setdefault((s["url"], name), (p, []))[1].append((r["uid"], int(cols[0]), r["segments"][0][0]))
            break

def width_job(item):
    (url, name), (p, us) = item
    return g.ap_widths(url, name, p["rate"], p["start"], p["chunks"][0], us)

with ThreadPoolExecutor(16) as ex:
    widths = pd.DataFrame([w for ws in ex.map(width_job, jobs.items()) for w in ws])
widths = widths.merge(units[["uid", "area", "subject"]], on="uid")
print(f"Units with a channel in a filtered series: {len(widths)} of {len(units)}, in {len(jobs)} file-series; widths measured: "
      f"{int(widths.width_ms.notna().sum())} ({time.time() - started:.0f} s)")
print("Spikes per unit used: median", int(widths.n_spikes.median()), ", min", int(widths.n_spikes.min()))
print(widths.groupby(["area", "subject"]).width_ms.agg(["mean", "std", "median", "size"]).round(3).to_string())
print(widths.groupby("area").width_ms.agg(["mean", "std", "median", "min", "max", "size"]).round(3).to_string())
w_gp, w_vl = widths[widths.area == "GPi"].width_ms.dropna(), widths[widths.area == "VLa"].width_ms.dropna()
res = stats.ranksums(w_gp, w_vl)
print(f"GPi {w_gp.mean():.3f} ms against VLa {w_vl.mean():.3f} ms; rank sum z = {res.statistic:.2f}, p = {res.pvalue:.2g}")
print("Sampling interval of the trace: %.4f ms" % (1000 / list(sessions[0]["processed"].values())[0]["rate"]))
fig, ax = plt.subplots(figsize=(5, 3))
ax.hist([w_gp, w_vl], bins=np.arange(0, 1.2, 0.041), label=["GPi", "VLa"])
ax.set_xlabel("trough to peak (ms)"); ax.set_ylabel("units"); ax.legend()
print("Units outside the axes (above 1.2 ms):", int((widths.width_ms > 1.2).sum()))

**Result: reproduced.** Trough-to-peak widths from the filtered traces average 0.247 ms in GPi and
0.509 ms in VLa (532 of 547 units measured), within 10% of 0.24 and 0.50 ms, and GPi widths are
smaller (rank sum p = 2.9e-37). The trace is sampled every 0.041 ms, and the 200 Hz high-pass filter
of the dandiset differs from the paper's 300 Hz.

## hold-period-ramps

> The firing rate of most units ramps during the start-position hold period (97% of GPi and 98% of
> VLa units with a significant linear trend); slopes are distributed symmetrically around zero,
> without a difference between GPi and VLa.

In [ ]:
def ramps(df):
    per_unit = df.groupby(["uid", "area"]).slope_sig.any().groupby("area").agg(["mean", "sum", "size"])
    return per_unit

for name, df in (("Dandiset rule", ud), ("Variant, the paper's records", ud_paper)):
    t = ramps(df)
    print(name, "- units with a significant control-period slope in at least one direction:")
    print(t.assign(percent=lambda d: (100 * d["mean"]).round(1)).to_string())
    gs, vs = df[df.area == "GPi"].slope_per_s2, df[df.area == "VLa"].slope_per_s2
    print(f"  mean slope (spikes/s2) GPi {gs.mean():.3f}, VLa {vs.mean():.3f}; medians {gs.median():.3f}, {vs.median():.3f}; "
          f"rank sum p = {stats.ranksums(gs, vs).pvalue:.3g}")
    sig = df[df.slope_sig]
    print(f"  significant unit-direction slopes positive: {(sig.slope_per_s2 > 0).mean():.1%} of {len(sig)}")
print("By animal (dandiset rule):")
print(ud.groupby(["uid", "area", "subject"]).slope_sig.any().groupby(["area", "subject"]).mean().round(3).to_string())
print("Variant, significance at alpha 0.01 (99% CI) instead of 0.05:")
def sig01(df):
    out = []
    for _, r in df.iterrows():
        cue = int(round(1500 - r.median_rt_ms))
        x = np.arange(cue - 700, cue)
        lr = stats.linregress(x, r.sdf[x])
        out.append(lr.pvalue < 0.01)
    return pd.Series(out, index=df.index)
ud["slope_sig01"] = sig01(ud)
print(ud.groupby(["uid", "area"]).slope_sig01.any().groupby("area").mean().round(3).to_string())

**Result: partially reproduced.** 98.5% of GPi and 98.6% of VLa units have a significant
control-period slope in at least one direction, within 10% of 97% and 98% (97.0% and 97.5% at
alpha 0.01), and 49.9% of significant slopes are positive. The slopes of GPi and VLa differ by the
rank sum test (mean -0.598 against 0.233 spikes/s2, p = 0.042), where the paper finds no difference
(means -0.05 and -0.14, p = 0.29); with the paper's records they do not differ (p = 0.19). The
criterion did not define a partial outcome; the first part holds and the second does not.

## mvt-responsive-fraction

> Nearly all units change their discharge around movement for at least one direction (GPi 99%;
> VLa 88.1% in Table 1, 94% in the text).

In [ ]:
def responsive(df):
    return df.assign(resp=df.form != "none").groupby(["uid", "area", "subject"]).resp.any()

for name, df in (("Dandiset rule", ud), ("Variant, the paper's records", ud_paper)):
    r = responsive(df)
    t = r.groupby("area").agg(["sum", "size", "mean"])
    print(name); print(t.assign(percent=lambda d: (100 * d["mean"]).round(1)).to_string())
    print(r.groupby(["area", "subject"]).mean().round(3).to_string())
print("\nUnit-directions with a response (dandiset rule):", (ud.form != "none").groupby(ud.area).agg(["sum", "size"]).to_dict("index"))
print("Variant, alpha 0.0001 instead of 0.001 (stricter):")
strict = []
for _, r in ud.iterrows():
    f_ = g.detect(r.sdf, r.median_rt_ms, alpha=0.0001)
    strict.append(np.isfinite(f_["onset"]))
print("  units with an increase in at least one direction:", ud.assign(s=strict).groupby(["uid", "area"]).s.any().groupby("area").mean().round(3).to_dict())

**Result: reproduced.** 98.5% of GPi units (264 of 268) and 97.5% of VLa units (272 of 279) respond
in at least one direction. GPi is within 10% of 99%. VLa is within 10% of the text's 94% but not
of Table 1's 88.1%; the criterion accepts either reading. With alpha 0.0001 the fractions with an
increase alone fall to 81.0% and 77.4%.

## increases-most-common

> Monophasic increases are the most common form of response in both GPi and VLa (48.7% and 65.3%
> of responses), and increases make up most detected changes (61.5% and 68.8%, biphasic responses
> counted twice), more than decreases.

In [ ]:
FORMS = ["increase", "decrease", "inc_first", "dec_first"]
def forms_table(df):
    r = df[df.form != "none"]
    t = pd.crosstab(r.area, r.form)[FORMS]
    pct = (100 * t.div(t.sum(axis=1), axis=0)).round(1)
    inc = t.increase + t.inc_first + t.dec_first
    changes = t.increase + t.decrease + 2 * (t.inc_first + t.dec_first)
    return t, pct, inc, changes

for name, df in (("Dandiset rule", ud), ("Variant, the paper's records", ud_paper)):
    t, pct, inc, changes = forms_table(df)
    print(name, "- responses by form (unit-directions):"); print(t.to_string()); print("percent:"); print(pct.to_string())
    for a in t.index:
        print(f"  {a}: increases {inc[a]} of {changes[a]} changes ({100 * inc[a] / changes[a]:.1f}%)")
    chi = stats.chi2_contingency(np.array([[inc[a], changes[a] - inc[a]] for a in t.index]))
    allinc, allchg = inc.sum(), changes.sum()
    print(f"  increases against decreases over both areas: {allinc} of {allchg}, binomial p = {stats.binomtest(int(allinc), int(allchg)).pvalue:.2g}; "
          f"GPi against VLa chi-squared p = {chi.pvalue:.3g}")
print("By animal (dandiset rule), percent monophasic increases:")
r = ud[ud.form != "none"]
print((100 * pd.crosstab([r.area, r.subject], r.form, normalize="index")).round(1)[FORMS].to_string())

**Result: partially reproduced.** Monophasic increase is the most common form in both areas (GPi
46.9%, VLa 50.9% of responses), and increases make up 58.8% of GPi and 60.2% of VLa changes (746 of
1254 overall, binomial p = 1.9e-11). The GPi values are within 10% of 48.7% and 61.5%; the VLa values
are below 65.3% and 68.8% by more than 10%, also with the paper's records (54.1% and 61.4%).

## monophasic-more-in-vla

> Monophasic changes are more common in VLa than in GPi (88.9% against 66.7% of responses), and
> polyphasic responses more common in GPi.

In [ ]:
for name, df in (("Dandiset rule", ud), ("Variant, the paper's records", ud_paper)):
    r = df[df.form != "none"]
    mono = r.form.isin(["increase", "decrease"])
    t = pd.crosstab(r.area, mono)
    chi = stats.chi2_contingency(t.to_numpy())
    print(name); print((100 * mono.groupby(r.area).mean()).round(1).to_string())
    print(f"  counts (polyphasic, monophasic): {t.to_dict('index')}; chi-squared p = {chi.pvalue:.2g}")
r = ud[ud.form != "none"]
print("By animal (dandiset rule):", (100 * r.form.isin(["increase", "decrease"]).groupby([r.area, r.subject]).mean()).round(1).to_dict())

**Result: not reproduced.** Monophasic responses are 71.0% in GPi and 76.7% in VLa; the difference
is not significant (chi-squared p = 0.051), and VLa is more than 10% below 88.9%. With the paper's
records the values are 69.0% and 81.2% (p = 0.00036), both within tolerance; the difference comes
with the selection of units.

## integrated-change-positive

> The mean integrated change in firing over the peri-movement epoch is positive in both GPi and VLa
> (3.89 and 1.75 spikes), significantly different from zero.

In [ ]:
for name, df in (("Dandiset rule", ud), ("Variant, the paper's records", ud_paper)):
    print(name)
    for a in ("GPi", "VLa"):
        x = df[df.area == a].integrated
        w = stats.wilcoxon(x)
        print(f"  {a}: mean {x.mean():.2f} spikes (SD {x.std():.2f}), median {x.median():.2f}, n {len(x)}; signed rank p = {w.pvalue:.2g}; "
              f"range {x.min():.1f} to {x.max():.1f}")
    print("  by animal:", df.groupby(["area", "subject"]).integrated.mean().round(2).to_dict())

**Result: partially reproduced.** The mean integrated change is positive in GPi (3.07 spikes, signed
rank p = 1.1e-10) and in VLa (1.50 spikes, p = 3.7e-18), but both are more than 10% below 3.89 and
1.75. With the paper's records they are 3.42 and 1.97 spikes.

## fig3a-vla-leads-gpi

> Peri-movement changes begin later in GPi than in VLa (median onsets -47 and -90 ms).

In [ ]:
def onsets(df, area, kind):
    col = {"inc": "inc_onset", "dec": "dec_onset"}
    if kind == "all":
        return pd.concat([df[df.area == area].inc_onset, df[df.area == area].dec_onset]).dropna()
    return df[df.area == area][col[kind]].dropna()

def compare(df, a1, k1, a2, k2, label):
    x, y = onsets(df, a1, k1), onsets(df, a2, k2)
    res = stats.ranksums(x, y)
    print(f"  {label}: {a1} {k1} median {x.median():.1f} ms (n {len(x)}), {a2} {k2} median {y.median():.1f} ms (n {len(y)}); "
          f"difference {x.median() - y.median():.1f} ms; rank sum z = {res.statistic:.2f}, p = {res.pvalue:.2g}")
    return x, y

for name, df in (("Dandiset rule", ud), ("Variant, the paper's records", ud_paper)):
    print(name)
    x, y = compare(df, "GPi", "all", "VLa", "all", "all changes")
for subj in ("Gaia", "Isis"):
    print("Dandiset rule,", subj)
    compare(ud[ud.subject == subj], "GPi", "all", "VLa", "all", "all changes")
x, y = onsets(ud, "GPi", "all"), onsets(ud, "VLa", "all")
fig, ax = plt.subplots(figsize=(5, 3))
for v, lab in ((x, "GPi"), (y, "VLa")):
    ax.step(np.sort(v), np.arange(1, len(v) + 1) / len(v), label=lab)
ax.set_xlabel("onset from movement onset (ms)"); ax.set_ylabel("cumulative fraction"); ax.legend()
print("Onset range (ms):", int(min(x.min(), y.min())), "to", int(max(x.max(), y.max())), "; all inside the axes")

**Result: not reproduced.** The median onset of all changes is -48 ms in GPi (641 changes) and -67 ms
in VLa (613); GPi is later by 19 ms, but not significantly (rank sum p = 0.079), and the VLa median
is outside -90 +/- 9 ms. In Gaia the medians are equal (-48.5 and -48 ms). With the paper's records
the medians are -49 and -85 ms (p = 0.004), which would meet the criterion; the difference comes
with the selection of units.

## fig3b-gpi-decr-lag-vla-incr

> GPi decreases lag behind VLa increases (median onsets -27 and -88 ms).

In [ ]:
for name, df in (("Dandiset rule", ud), ("Variant, the paper's records", ud_paper)):
    print(name)
    compare(df, "GPi", "dec", "VLa", "inc", "GPi decreases against VLa increases")
for subj in ("Gaia", "Isis"):
    print("Dandiset rule,", subj)
    compare(ud[ud.subject == subj], "GPi", "dec", "VLa", "inc", "GPi decreases against VLa increases")

**Result: reproduced.** GPi decreases begin at a median of -29.5 ms (264) and VLa increases at -84 ms
(369), both within tolerance of -27 and -88 ms, and GPi decreases are later (rank sum p = 0.00021).
The lag holds in Gaia (0 against -95 ms, p = 0.0014) and in Isis without significance (-44 against
-81 ms, p = 0.055), and with the paper's records (-36 against -85 ms, p = 0.00069).

## fig3c-vla-decr-lead-gpi-incr

> VLa decreases precede GPi increases (median onsets -108 and -62 ms).

In [ ]:
for name, df in (("Dandiset rule", ud), ("Variant, the paper's records", ud_paper)):
    print(name)
    compare(df, "GPi", "inc", "VLa", "dec", "GPi increases against VLa decreases")
for subj in ("Gaia", "Isis"):
    print("Dandiset rule,", subj)
    compare(ud[ud.subject == subj], "GPi", "inc", "VLa", "dec", "GPi increases against VLa decreases")

**Result: not reproduced.** GPi increases begin at a median of -69 ms (377) and VLa decreases at -19
ms (244); VLa decreases do not lead (rank sum p = 0.17), and the VLa median is far from -108 ms. In
Gaia VLa decreases come later than GPi increases (12.5 against -75 ms, p = 0.0028). With the paper's
records VLa decreases lead by 13 ms without significance (-86 against -73 ms, p = 0.2).

## gating-conflict-fraction

> Comparing every GPi response with every VLa response, 96% of comparisons conflict with gating;
> only 4% have a GPi onset 0 to 50 ms before the VLa onset with the opposite sign.

In [ ]:
def gating(df, lo=0, hi=50):
    def changes(a):
        d = df[df.area == a]
        return np.r_[d.inc_onset.dropna(), d.dec_onset.dropna()], np.r_[np.ones(d.inc_onset.notna().sum()), -np.ones(d.dec_onset.notna().sum())]
    gt, gs = changes("GPi")
    vt, vs = changes("VLa")
    lead = vt[None, :] - gt[:, None]
    fits = (lead >= lo) & (lead <= hi) & (gs[:, None] != vs[None, :])
    return fits.size, int(fits.sum())

for name, df in (("Dandiset rule", ud), ("Variant, the paper's records", ud_paper)):
    n, fit = gating(df)
    print(f"{name}: {n} comparisons, {fit} fit gating; conflicting {100 * (1 - fit / n):.1f}%")
for lo, hi in ((0, 20), (0, 100), (0, 200)):
    n, fit = gating(ud, lo, hi)
    print(f"  variant, GPi leading by {lo} to {hi} ms: conflicting {100 * (1 - fit / n):.1f}%")
n, fit = gating(ud)
rng = np.random.default_rng(1)
d = ud.copy()
null = []
for _ in range(200):
    d["inc_onset"] = rng.permutation(ud.inc_onset.to_numpy())
    d["dec_onset"] = rng.permutation(ud.dec_onset.to_numpy())
    nn, ff = gating(d)
    null.append(100 * (1 - ff / nn))
print(f"Reference: onsets shuffled across unit-directions, conflicting {np.mean(null):.1f}% (2.5 to 97.5 percentile {np.percentile(null, 2.5):.1f} to {np.percentile(null, 97.5):.1f})")

**Result: reproduced.** 95.4% of the 392,933 GPi-VLa comparisons conflict with gating (96 +/- 9.6%);
95.6% with the paper's records. The number says little: with onsets shuffled across
unit-directions it is 95.1% (95.0 to 95.2), and it follows the width of the gating window (98.0%
for 0 to 20 ms, 86.6% for 0 to 200 ms).

## directional-units

> A majority of GPi neurons (109 of 206) and some VLa neurons (56 of 179) respond significantly more
> strongly to one movement direction.

In [ ]:
def directional(df, records):
    seg = {r["uid"]: r["segments"] for r in records}
    out = []
    for uid, d in df[df.form != "none"].groupby("uid"):
        first = np.nanmin(np.r_[d.inc_onset, d.dec_onset]) / 1000.0
        c = {}
        for target in ("Left", "Right"):
            vals = []
            for spikes, t in seg[uid]:
                tt = t[t.valid & (t.target == target)]
                vals.append(window_counts(spikes, tt.movement_start_time.to_numpy(), first, first + 0.3))
            c[target] = np.concatenate(vals)
        p = stats.ranksums(c["Left"], c["Right"]).pvalue if min(len(c["Left"]), len(c["Right"])) else np.nan
        out.append({"uid": uid, "area": d.area.iloc[0], "subject": d.subject.iloc[0], "p": p})
    return pd.DataFrame(out)

for name, df, recs in (("Dandiset rule", ud, dand_records), ("Variant, the paper's records", ud_paper, paper_records)):
    dd = directional(df, recs)
    t = dd.assign(dir=dd.p < 0.005).groupby("area").dir.agg(["sum", "size", "mean"])
    print(name); print(t.assign(percent=lambda x: (100 * x["mean"]).round(1)).to_string())
    chi = stats.chi2_contingency(pd.crosstab(dd.area, dd.p < 0.005).to_numpy())
    print(f"  GPi against VLa chi-squared p = {chi.pvalue:.2g}")
    if name == "Dandiset rule":
        print("  by animal:", dd.assign(dir=dd.p < 0.005).groupby(["area", "subject"]).dir.mean().round(3).to_dict())
        print("  variant p < 0.05:", dd.assign(dir=dd.p < 0.05).groupby("area").dir.mean().round(3).to_dict())

**Result: partially reproduced.** GPi units are directional more often than VLa units (46.2%, 122 of
264, against 24.3%, 66 of 272; chi-squared p = 1.7e-07), but both fractions are more than 10% below
52.9% and 31.3%. With the paper's records they are 48.2% and 26.8%.

## Pairs of simultaneously recorded GPi and VLa units

Pairs are the GPi and VLa units of the dandiset rule in the same file. Windows: rest is 1.5 to 1.0 s
before movement onset, movement 0.2 s before to 0.3 s after, in valid trials.

In [ ]:
pairs = []
for s in sessions:
    us = units[units.session == s["session_id"]]
    gps, vls = us[us.area == "GPi"], us[us.area == "VLa"]
    for _, a in gps.iterrows():
        for _, b in vls.iterrows():
            pairs.append({"session": s["session_id"], "subject": s["subject"], "gpi": a.uid, "vla": b.uid})
pairs = pd.DataFrame(pairs)
spk = {r["uid"]: r["segments"][0][0] for r in dand_records}
sess_of = {s["session_id"]: s for s in sessions}
print(f"GPi-VLa pairs in the same file: {len(pairs)} in {pairs.session.nunique()} files")
print(pairs.groupby("subject").size().to_string())

## fig4-ccf-rare

> Short-time-scale spike-to-spike correlations between simultaneously recorded GPi and VLa units are
> rare (3.5% of 427 pairs over whole recordings, 2.2% of 183 at rest, 5.2% of 172 during movement)
> and their peaks are not larger than those of jittered controls.

In [ ]:
started = time.time()
rng = np.random.default_rng(4)
N_SURR = 400
ccf_rows, ctrl_peaks = [], {"whole": [], "rest": [], "movement": []}
for _, p in pairs.iterrows():
    s = sess_of[p.session]
    a, b = spk[p.gpi], spk[p.vla]
    t = s["vt"][s["vt"].valid]
    periods = {"whole": None,
               "rest": np.c_[t.movement_start_time + g.REST_WIN[0], t.movement_start_time + g.REST_WIN[1]],
               "movement": np.c_[t.movement_start_time + g.MVT_WIN[0], t.movement_start_time + g.MVT_WIN[1]]}
    for name, win in periods.items():
        if win is None:
            x, y = a, b
            dur = max(a[-1], b[-1]) - min(a[0], b[0])
        else:
            x, y = a[g._in_windows(a, win)], b[g._in_windows(b, win)]
            dur = float(np.sum(win[:, 1] - win[:, 0]))
        if min(len(x), len(y)) < 100:
            continue
        r = g.ccf_test(x, y, dur, rng, N_SURR)
        ccf_rows.append({"period": name, "subject": p.subject, "peak": r["peak"], "sign": r["sign"],
                         "thr95": r["thr95"], "significant": r["significant"]})
        ctrl_peaks[name].append(r["ctrl_peaks"])
cc = pd.DataFrame(ccf_rows)
print(f"CCFs computed in {time.time() - started:.0f} s; {N_SURR} jittered controls per pair and period")
for name in ("whole", "rest", "movement"):
    d = cc[cc.period == name]
    k, n = int(d.significant.sum()), len(d)
    ks_p = g.ks_permutation(d.peak.to_numpy(), np.vstack(ctrl_peaks[name]))
    print(f"{name}: {n} pairs; significant {k} ({100 * k / n:.1f}%), binomial p against 5% (greater) = "
          f"{stats.binomtest(k, n, 0.05, alternative='greater').pvalue:.3g}; signs of significant peaks "
          f"+{int((d[d.significant].sign > 0).sum())} / -{int((d[d.significant].sign < 0).sum())}; "
          f"KS permutation p = {ks_p:.3f}; median peak {d.peak.median():.3f}, median control peak {np.median(np.vstack(ctrl_peaks[name])):.3f}, "
          f"median control 95th percentile {d.thr95.median():.3f}")
print("By animal, percent significant:", (100 * cc.groupby(["period", "subject"]).significant.mean()).round(1).to_dict())

**Result: not reproduced.** By the paper's test the peak CCFs of the 338 whole-recording pairs are
larger than those of the jittered controls (KS permutation p = 0.003), and so are those of the 306
movement pairs (p = 0.045); the 298 rest pairs are not (p = 0.705). Significant pairs are few (5.6%,
3.4% and 6.5%, none above 5% by a binomial test, p 0.14 to 0.93), as the paper reports, and their
peaks are mostly positive (15 of 19 whole-recording, 15 of 20 movement), where gating predicts
negative ones. The effect is small: median peak 0.042 against a control median of 0.038.

## fig4-noise-corr-rare

> Trial-to-trial noise correlations between GPi and VLa are rare (5.4% of 332 pairs at rest, 7.2% of
> 372 during movement), not different from shuffled controls, and equally often positive and
> negative.

In [ ]:
rng = np.random.default_rng(5)
nc_rows, nc_ctrl = [], {"rest": [], "movement": []}
for _, p in pairs.iterrows():
    s = sess_of[p.session]
    for target in ("Left", "Right"):
        t = s["vt"][s["vt"].valid & (s["vt"].target == target)]
        al = t.movement_start_time.to_numpy()
        for name, (lo, hi) in (("rest", g.REST_WIN), ("movement", g.MVT_WIN)):
            ca = window_counts(spk[p.gpi], al, lo, hi).astype(float)
            cb = window_counts(spk[p.vla], al, lo, hi).astype(float)
            if ca.sum() < 10 or cb.sum() < 10 or len(al) < 10 or ca.std() == 0 or cb.std() == 0:
                continue
            za, zb = stats.zscore(ca), stats.zscore(cb)
            ok = (np.abs(za) <= 3) & (np.abs(zb) <= 3)
            ca, cb = ca[ok], cb[ok]
            if ca.std() == 0 or cb.std() == 0:
                continue
            r = np.corrcoef(ca, cb)[0, 1]
            sh = np.array([np.corrcoef(ca, rng.permutation(cb))[0, 1] for _ in range(400)])
            nc_rows.append({"period": name, "pair": f"{p.gpi}|{p.vla}", "subject": p.subject, "target": target, "r": r})
            nc_ctrl[name].append(sh)
nc = pd.DataFrame(nc_rows)
for name in ("rest", "movement"):
    d = nc[nc.period == name].copy()
    ctrl = np.vstack(nc_ctrl[name])
    lo, hi = np.percentile(ctrl, [2.5, 97.5])
    d["sig"] = (d.r < lo) | (d.r > hi)
    pos, neg = int((d.r > hi).sum()), int((d.r < lo).sum())
    ks_p = g.ks_permutation(np.abs(d.r.to_numpy()), np.abs(ctrl))
    by_pair = d.groupby("pair").sig.any()
    print(f"{name}: (a) {len(d)} pair-targets, significant {int(d.sig.sum())} ({100 * d.sig.mean():.1f}%), +{pos} / -{neg} "
          f"(binomial p of the sign balance {stats.binomtest(pos, pos + neg).pvalue:.3g}); "
          f"(b) {len(by_pair)} pairs, significant in either target {int(by_pair.sum())} ({100 * by_pair.mean():.1f}%)")
    print(f"   KS permutation p (absolute correlations against shuffles) = {ks_p:.3f}; median r {d.r.median():.3f}; "
          f"shuffle band {lo:.3f} to {hi:.3f}")

**Result: not reproduced.** Absolute noise correlations exceed those of trial-shuffled controls at
rest and during movement (KS permutation p < 0.001 in both). 10.1% (rest) and 10.8% (movement) of
pair-targets lie outside the shuffled 2.5 to 97.5 percentiles, against 5.4% and 7.2%, or 17.5% and
18.6% of pairs counting either target. Positive and negative correlations are about equally common
(38 and 29 at rest, p = 0.33; 39 and 33 during movement, p = 0.56), as the paper reports.

## fig5-vla-decrease-after-gpi-bursts

> At rest, GPi burst onsets, burst offsets and pause onsets are each followed by a small, slow
> decrease of the population VLa firing rate.

In [ ]:
started = time.time()
rng = np.random.default_rng(6)
LAGS = np.arange(-800, 801)
ta = {"burst onset": [], "burst offset": [], "pause onset": []}
ctrl_sum = {k: 0.0 for k in ta}
burst_cache, burst_stats = {}, []
pair_sig = {k: [] for k in ta}
n_excluded = {k: 0 for k in ta}
n_events = {k: [] for k in ta}
for _, p in pairs.iterrows():
    s = sess_of[p.session]
    t = s["trials"]
    end = t["return_stop_time"] if "return_stop_time" in t else t["stop_time"]
    task = np.c_[t.movement_start_time + g.TASK_EXCL[0], end.fillna(t.stop_time) + g.TASK_EXCL[1]]
    task = task[np.isfinite(task).all(1)]
    a, b = spk[p.gpi], spk[p.vla]
    T = int(np.ceil(max(a[-1], b[-1]) * 1000)) + 2000
    in_task = np.zeros(T, bool)
    for lo_, hi_ in (task * 1000).astype(int):
        in_task[max(lo_, 0):min(hi_, T)] = True
    if p.gpi not in burst_cache:
        rest_a = a[~in_task[(a * 1000).astype(int)]]
        rest_dur = (~in_task[int(a[0] * 1000):int(a[-1] * 1000)]).sum() / 1000
        rate = len(rest_a) / rest_dur
        edges = np.flatnonzero(np.diff(np.r_[0, (~in_task).astype(int), 0]))
        bl, pl = [], []
        for lo_, hi_ in edges.reshape(-1, 2) / 1000:      # rest segments
            x = a[(a >= lo_) & (a < hi_)]
            bl += g.bursts(x, rate)
            pl += g.pauses(x, rate)
        burst_cache[p.gpi] = (bl, pl)
        burst_stats.append({"gpi": p.gpi, "rest_s": rest_dur, "rate": rate, "n_bursts": len(bl), "n_pauses": len(pl),
                            "burst_ms": np.mean([1000 * (e - b0) for b0, e in bl]) if bl else np.nan,
                            "pause_ms": np.mean([1000 * (e - b0) for b0, e in pl]) if pl else np.nan})
    bl, pl = burst_cache[p.gpi]
    events = {"burst onset": np.array([e[0] for e in bl]), "burst offset": np.array([e[1] for e in bl]),
              "pause onset": np.array([e[0] for e in pl])}
    counts = np.bincount((b * 1000).astype(int), minlength=T)[:T].astype(float)
    rest = (~in_task).astype(float)
    vrate = gaussian_filter1d(counts * 1000.0, 10.0) * rest       # VLa rate, rest samples only
    for k, ev in events.items():
        ev = ev[(ev > 1.0) & (ev < T / 1000 - 1.0)]
        if len(ev) < 5:
            continue
        idx = (ev * 1000).astype(int)
        X, W = g.event_sums(vrate, idx), g.event_sums(rest, idx)
        ratio = np.where(W > 0.5, X / np.maximum(W, 0.5), np.nan)     # lags where no event has a rest sample: NaN
        M = len(ratio)
        avg = ratio[LAGS % M]
        if np.isnan(avg).any():
            n_excluded[k] += 1
            continue
        base = avg[(LAGS >= -800) & (LAGS < -50)]
        lo_b, hi_b = np.percentile(base, [2.5, 97.5])
        if hi_b - lo_b > 0.6 * abs(base.mean()):
            n_excluded[k] += 1
            continue
        ta[k].append(avg)
        n_events[k].append(len(idx))
        post = avg[(LAGS >= 0) & (LAGS <= 100)]
        pair_sig[k].append("decrease" if (post < lo_b).any() else ("increase" if (post > hi_b).any() else "none"))
        shifts = rng.integers(2000, 20001, size=400) * rng.choice([-1, 1], size=400)
        c_ = np.array([ratio[(LAGS + sh) % M] for sh in shifts])
        ctrl_sum[k] = ctrl_sum[k] + np.where(np.isnan(c_), np.nanmean(avg), c_)
print(f"Bursts and pauses in {time.time() - started:.0f} s")
bs = pd.DataFrame(burst_stats)
print(f"GPi units: {len(bs)}; rest per unit median {bs.rest_s.median():.0f} s; bursts per minute of rest median "
      f"{(60 * bs.n_bursts / bs.rest_s).median():.2f}; pauses per minute median {(60 * bs.n_pauses / bs.rest_s).median():.2f}")
print(f"Mean burst duration {np.nanmean(bs.burst_ms):.1f} ms (paper 24 ms); mean pause duration {np.nanmean(bs.pause_ms):.1f} ms (paper 195 ms)")
fig, axes = plt.subplots(1, 3, figsize=(12, 3), sharey=True)
win = (LAGS >= 0) & (LAGS <= 300)
for ax, k in zip(axes, ta):
    pop = np.mean(ta[k], 0)
    c = ctrl_sum[k] / len(ta[k])                   # control population averages, 400 x lags
    lo_c, hi_c = np.percentile(c, [2.5, 97.5], axis=0)
    shift = np.percentile((c[:, win] - lo_c[win]).min(1), 5)
    lo_corr = lo_c + min(shift, 0)
    below = win & (pop < lo_corr)
    above = win & (pop > hi_c - min(shift, 0))
    first = LAGS[below][0] if below.any() else None
    sig = pd.Series(pair_sig[k]).value_counts()
    print(f"{k}: {len(ta[k])} pairs (excluded as noisy {n_excluded[k]}), events per pair median {np.median(n_events[k]):.0f}; "
          f"population rate at baseline {pop[LAGS < -50].mean():.2f} spikes/s, minimum in 0 to 300 ms {pop[win].min():.2f} "
          f"({100 * (pop[win].min() / pop[LAGS < -50].mean() - 1):.1f}%); below the corrected band: {bool(below.any())}, "
          f"from {first} ms; above it: {bool(above.any())}")
    print(f"   pairs with a change in 0 to 100 ms: {int(sig.drop('none', errors='ignore').sum())} of {len(pair_sig[k])} "
          f"({100 * sig.drop('none', errors='ignore').sum() / len(pair_sig[k]):.1f}%); decreases {int(sig.get('decrease', 0))}, increases {int(sig.get('increase', 0))}")
    ax.plot(LAGS, pop, "k"); ax.fill_between(LAGS, lo_corr, hi_c - min(shift, 0), color="0.8")
    ax.set_title(k); ax.set_xlabel("ms from event")
axes[0].set_ylabel("VLa rate (spikes/s)")
print("All plotted values are inside the axes.")

**Result: not reproduced.** The population VLa rate does not fall below the control band in 0 to
300 ms after GPi burst onsets (150 pairs; lowest point 2.5% below baseline), burst offsets (150;
1.9%) or pause onsets (110; 1.9%). Per pair, 50.7% show a change in 0 to 100 ms after burst onsets,
43 decreases and 33 increases, against 11.6% in the paper, so this per-pair test is not the
paper's. Bursts here last 67.1 ms on average and pauses 145.3 ms, against 24 and 195 ms in the
paper, so the burst detection also differs from the paper's.

## behavior-rt-mt

> Reaction times do not differ between the animals or reach directions; movement durations are
> longer for reaches to the right target, and NHP G moves more slowly than NHP I.

In [ ]:
v = tr[tr.valid]
per = v.groupby(["subject", "session", "target"]).agg(rt=("rt", "median"), mt=("mt", "median"), n=("rt", "size")).reset_index()
print("Per file and direction, medians (s):")
print(per.groupby(["subject", "target"])[["rt", "mt"]].agg(["median", "count"]).round(3).to_string())
for col in ("rt", "mt"):
    d = per.dropna(subset=[col])
    model = ols(f"{col} ~ C(subject) * C(target)", data=d).fit()
    print(f"\nTwo-way ANOVA of {col} (type II), {len(d)} file-directions:")
    print(sm.stats.anova_lm(model, typ=2).round(4).to_string())
print("\nVariant, trials pooled: RT animal p = %.3g, direction p = %.3g" % (
    stats.ranksums(v[v.subject == "Gaia"].rt, v[v.subject == "Isis"].rt).pvalue,
    stats.ranksums(v[v.target == "Left"].rt, v[v.target == "Right"].rt).pvalue))

**Result: partially reproduced.** Movement durations are longer to the right target (Gaia 0.273
against 0.218 s, Isis 0.204 against 0.175 s) and longer in Gaia, both with p < 1e-4 in the
two-way ANOVA. Reaction times differ between the animals (F = 15.19, p = 0.0001; medians Gaia 0.243 and
0.242 s, Isis 0.235 and 0.241 s), where the paper finds no difference, and not between directions (p = 0.073).
Movement durations exist for only 32 of the 119 Gaia files.

## figs3-emg-onset

Blocked. The EMG is not in the dandiset; it is in the paper's Dryad deposit (S3_EMGdata files).
Neither the Methods nor `FigS3.m`, which only plots the trial-averaged EMG, say how the onset of
the EMG change was determined, so the reported onsets cannot be recomputed by the paper's method.

## fig1-vla-localization

Blocked. The stimulation mapping (responses to GPi, SCP and motor cortex stimulation along the
electrode tracks) is not in the dandiset, which holds the task recordings only, nor in the Dryad
deposit.

## fig6-simulation

Blocked. The simulation uses no recorded data. The Dryad deposit holds its output
(`Data_Fig6.mat`) and a plotting script, but not the model thalamocortical cell or the code that
generated correlated GPi spike trains; reimplementing the model of the paper's reference 103 was
not attempted.

In [ ]:
print(f"Notebook ran in {(time.time() - NB_START) / 60:.1f} min")

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

| Claim | Data | Outcome |
|---|---|---|
| unit-link | with inputs | reproduced |
| table1-unit-counts | dandiset | not reproduced |
| table1-rest-rate | dandiset | reproduced |
| table1-ap-width | dandiset | reproduced |
| hold-period-ramps | dandiset | partially reproduced |
| mvt-responsive-fraction | dandiset | reproduced |
| increases-most-common | dandiset | partially reproduced |
| monophasic-more-in-vla | dandiset | not reproduced |
| integrated-change-positive | dandiset | partially reproduced |
| fig3a-vla-leads-gpi | dandiset | not reproduced |
| fig3b-gpi-decr-lag-vla-incr | dandiset | reproduced |
| fig3c-vla-decr-lead-gpi-incr | dandiset | not reproduced |
| gating-conflict-fraction | dandiset | reproduced |
| directional-units | dandiset | partially reproduced |
| fig4-ccf-rare | dandiset | not reproduced |
| fig4-noise-corr-rare | dandiset | not reproduced |
| fig5-vla-decrease-after-gpi-bursts | dandiset | not reproduced |
| behavior-rt-mt | dandiset | partially reproduced |
| figs3-emg-onset | elsewhere | blocked: methods underspecified |
| fig1-vla-localization | dandiset | blocked: data missing |
| fig6-simulation | elsewhere | blocked: other |

**How the analysis differs from the paper's.** The units are selected from the dandiset's own
labels, and each file's unit is counted once, because the dandiset does not say which units the
paper kept nor which units in consecutive files are one neuron; this gives about 40% more units
than the paper. The paper's own unit list, linked through the Dryad files, is shown as a variant;
with it, monophasic-more-in-vla and fig3a-vla-leads-gpi would meet their criteria, so those two
outcomes depend on the selection. The detection of responses follows the Methods and the paper's
detection function, but the detrending and the ISI function are implemented here; on the paper's
units they agree with its stored results for 91% (increases) and 81% (decreases) of
unit-directions. Trial exclusion follows the Methods and matches the paper's trial counts for 63%
of unit-directions. Widths are measured on the dandiset's 200 Hz filtered traces. The CCF jitters
the VLa spikes and does not apply the paper's sensitivity exclusion (c0 > 0.3). Burst and pause
detection is a reimplementation of the surprise method and gives longer bursts than the paper's;
its control shifts events within rest. A first run, not shown here, used log10 in the surprise and
gave bursts far longer than the paper's; the natural log of the paper's MATLAB is used.

**What was found about the dandiset.** It holds the paper's pre-MPTP GPi and VLa recordings: 364
of the 394 units the paper analysed are among its curated units, and their spike times reproduce
the paper's per-unit activity. It does not identify which units the paper analysed or link a neuron
across consecutive files. Location labels vary in case, 54 Gaia thalamic units are labelled only
"Thalamus" and 33 Isis units have no label or channel; 30 of the paper's units are missing from the
curated tables (some files have none). Every `session_start_time` is two days after the date in
the `session_id`. Error trials are flagged only in files with an `error_onset_time` column (84 and 43
files in the column table), and movement ends are in only 32 of the 119 Gaia files. One pre-MPTP file has no trials, raw data or
curated units, and one M1 spike train is stored twice. The EMG and the stimulation mapping are not
in the dandiset.